In [ ]:
import keras
import numpy as np
path = keras.utils.get_file('nietzsche.txt', origin='https://s3.amazonaws.com/text-datasets/nietzsche.txt')

text = open(path).read().lower()
print('Corpus length: ', len(text))

600901/600901 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Corpus length:  600893


In [ ]:
maxlen = 60
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i: i + maxlen])
  next_chars.append(text[i + maxlen])

print('Number of sequences: ', len(sentences))
chars = sorted(list(set(text)))
print('Unique characters: ', len(chars))
char_indices = dict((char, chars.index(char)) for char in chars)

print('Vectorization...')

x = np.zeros((len(sentences), maxlen, len(chars)), dtype=np.bool)
y = np.zeros((len(sentences), len(chars)), dtype=np.bool)

for i, sentence in enumerate(sentences):
  for t, char in enumerate(sentence):
    x[i, t, char_indices[char]] = 1
  y[i, char_indices[next_chars[i]]] = 1



Number of sequences:  200278
Unique characters:  57
Vectorization...


In [ ]:
from keras import layers
model = keras.models.Sequential()
model.add(keras.Input(shape=(maxlen, len(chars))))
model.add(layers.LSTM(128))
model.add(layers.Dense(len(chars), activation = 'softmax'))

In [ ]:
optimizer = keras.optimizers.RMSprop(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer = optimizer)

In [ ]:
def sample(preds, temperature=1.0):
  preds = np.asarray(preds).astype('float64')
  preds = np.log(preds) / temperature
  exp_preds = np.exp(preds)
  preds = exp_preds / np.sum(exp_preds)
  probas = np.random.multinomial(1, preds, 1)
  return np.argmax(probas)

In [ ]:
def reweight_distribution(original_distribution, temperature=0.5):
  distribution = np.log(original_distribution) / temperature
  distribution = np.exp(distribution)
  return distribution / np.sum(distribution)

In [ ]:
import random
import sys

for epoch in range(1, 60):
  print('epoch', epoch)
  model.fit(x, y, batch_size=128, epochs=1)

  start_index = random.randint(0, len(text) - maxlen - 1)
  generated_text = text[start_index: start_index + maxlen]
  print(' ---- Generating with seed: "'+generated_text+'"')
  for temperature in[0.2, 0.5, 1.0, 1.2]:
    print('----- temperature: ', temperature)
    sys.stdout.write(generated_text)

    for i in range(400):
      sampled = np.zeros((1, maxlen, len(chars)))
      for t, char in enumerate(generated_text):
        sampled[0, t, char_indices[char]] = 1.0

      preds = model.predict(sampled, verbose=0)[0]
      next_index = sample(preds, temperature)
      next_char = chars[next_index]

      generated_text += next_char
      generated_text = generated_text[1:]
      sys.stdout.write(next_char)

epoch 1
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 1.5359
 ---- Generating with seed: " whose perfection it pertains that he knows how to
appear,--"
----- temperature:  0.2
 whose perfection it pertains that he knows how to
appear,--in the sense conscience of the sense and all the same and all the sense the seldection and person to the delicate and the sense to the destrained and not be the sense the sense and the more and the sense the same something and seeks and the sense and in the sense and the sense of the sense the sense and the sense something to the self are seek to the sense the sense in the same the sense and somet----- temperature:  0.5
 seek to the sense the sense in the same the sense and something and her seek to the same present to and althyse of experied and plays." and believed the long the powassed as the deed and all the prefersions and same person with the conternely the most even in the prefersed to the merely art of he has a he does not with one has ven o

KeyboardInterrupt: 